# Recover-LoRA end-to-end: Qwen1.5-MoE-A2.7B (tier-3 test)

Trains per-expert Recover-LoRA adapters against the **exact deployed GGUF
quantization** and exports an `.lra` that loom serves with
`loom gguf run --recover-lora`. Runtime: **A100 (40 GB)** — Runtime →
Change runtime type → A100. Expected: ~1.5–2.5 h, ~25 CU (~$4).

Docs: [docs/TRAINING.md](https://github.com/ch4r10t33r/loom/blob/main/docs/TRAINING.md)

In [ ]:
# 0) GPU + disk sanity: this run needs a 40 GB-class card
import torch, shutil
assert torch.cuda.is_available(), "no GPU runtime selected"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(name, f"{vram:.0f} GB VRAM")
assert vram > 35, f"{name} has {vram:.0f} GB; select the A100 runtime (bf16 base is ~29 GB)"
print("disk free:", shutil.disk_usage("/").free // 2**30, "GiB")

In [ ]:
# 1) install loomtrain from the release tag + the house-pinned training stack
%pip -q install "loomtrain @ git+https://github.com/ch4r10t33r/loom@v0.45.0#subdirectory=train"
%pip -q install "transformers==4.55.4" "accelerate>=1.0,<2" "datasets>=3.0,<4" "gguf>=0.10"
import transformers; assert transformers.__version__ == "4.55.4"
!loomtrain --help

In [ ]:
# 2) fetch the deployed-quant GGUF (the adapters must learn THIS rounding)
from huggingface_hub import snapshot_download
import glob
snapshot_download("Qwen/Qwen1.5-MoE-A2.7B-Chat-GGUF", allow_patterns=["*q4_k_m*"], local_dir="gguf")
GGUF = glob.glob("gguf/**/*q4_k_m*.gguf", recursive=True)[0]
print("GGUF:", GGUF)

In [ ]:
# 3) train (mini-run: rank 4, 2M tokens). Watch for the nonzero
# 'first-step grad sum' line -- silence there means the backward was eaten
# and the run aborts loudly rather than training a ghost.
!python3 -u -m loomtrain.recover_lora train \
    --model Qwen/Qwen1.5-MoE-A2.7B-Chat --gguf "$GGUF" \
    --rank 4 --tokens 2000000 --batch 2 --seq 512 \
    --out a27b-rlora.pt 2>&1 | tee train.log

In [ ]:
# 4) export to LRA1 and fingerprint it
!python3 -m loomtrain.recover_lora export --ckpt a27b-rlora.pt --out a27b.lra
!ls -la a27b.lra && sha256sum a27b.lra

In [ ]:
# 5) keep the artifacts: copies to Drive (survives the VM) + browser download
from google.colab import drive, files
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/loom-artifacts
!cp a27b.lra a27b-rlora.pt train.log /content/drive/MyDrive/loom-artifacts/
print("saved to Drive: loom-artifacts/")
files.download('a27b.lra')

**Done.** Get `a27b.lra` (plus `train.log`) back to the loom repo owner's
machine — the serve-side verification runs there:

```sh
loom gguf run <a27b q4_k_m gguf> --recover-lora a27b.lra --prompt "..." --max-tokens 32 --temp 0
```

The engine validates the LRA1 header against the model and refuses shape
mismatches; flag off = byte-identical baseline for the A/B.